# TRF3 — Tribunal Regional Federal da 3ª Região

Public process consultation for the federal courts of the third region (SP
and MS), through the JSON API behind the PJe public consultation apps:
[pje1g-consultapublica.trf3.jus.br](https://pje1g-consultapublica.trf3.jus.br/)
(first instance, `cpopg`) and
[pje2g-consultapublica.trf3.jus.br](https://pje2g-consultapublica.trf3.jus.br/)
(appeals, `cposg`). Case-law search (`cjsg`) uses the separate portal at
[web.trf3.jus.br/jurisprudencia](https://web.trf3.jus.br/jurisprudencia/).

| Feature | Available |
|---------|-----------|
| cpopg   | Yes       |
| cposg   | Yes       |
| cjsg    | Yes       |
| cjpg    | No        |

## Looking up a single process

In [1]:
import juscraper as jus

trf3 = jus.scraper("trf3")

In [2]:
df = trf3.cpopg("5003536-21.2025.4.03.6342")
print(df.shape)
df[["id_cnj", "processo", "classe", "data_distribuicao", "orgao_julgador"]]

TRF3 cpopg:   0%|          | 0/1 [00:00<?, ?it/s]

TRF3 cpopg: 100%|██████████| 1/1 [00:04<00:00,  4.87s/it]

TRF3 cpopg: 100%|██████████| 1/1 [00:04<00:00,  4.87s/it]

(1, 14)


,id_cnj,processo,classe,data_distribuicao,orgao_julgador
0,50035362120254036342,5003536-21.2025.4.03.6342,PROCEDIMENTO DO JUIZADO ESPECIAL CÍVEL (436),24/09/2025,1ª Vara Gabinete JEF de Barueri


## Available columns

In [3]:
df.columns.tolist()

['id_cnj',
 'processo',
 'classe',
 'assunto',
 'data_distribuicao',
 'orgao_julgador',
 'orgao_julgador_colegiado',
 'jurisdicao',
 'endereco_orgao',
 'polo_ativo',
 'polo_passivo',
 'outros_interessados',
 'movimentacoes',
 'documentos']

The scalar columns come first; `polo_ativo`, `polo_passivo`,
`outros_interessados`, `movimentacoes` and `documentos` are lists of dicts.
Paginated sections (movements are served 15 per page) are fetched in full.

## Inspecting movements

In [4]:
movs = df.iloc[0]["movimentacoes"]
print(f"{len(movs)} events recorded")
for m in movs[:3]:
    print(f"  {m['data']} - {m['descricao']}")

70 events recorded
  24/09/2026 11:38:16 - Juntada de Petição de recurso inominado
  23/09/2026 13:25:54 - Juntada de Petição de ciência
  23/09/2026 10:23:38 - Expedição de Outros documentos.


## Inspecting parties

In [5]:
for polo in ["polo_ativo", "polo_passivo"]:
    print(polo)
    for p in df.iloc[0][polo]:
        print(f"  - {p['nome']} ({p['tipo']})")

polo_ativo
  - CINTIA REGIANE CORREA DOS SANTOS (AUTOR)
  - FERNANDO VANCETTI DA SILVA (ADVOGADO)
  - PAULO LEANDRO FAUSTINO (ADVOGADO)
polo_passivo
  - INSTITUTO NACIONAL DO SEGURO SOCIAL - INSS (REU)


## Appeals (`cposg`)

`cposg` takes the same arguments and returns the same columns, querying the
second-instance app. An appeal usually keeps the CNJ number of the original
case; `orgao_julgador_colegiado` carries the panel (turma).

In [6]:
df_2g = trf3.cposg("5021122-65.2018.4.03.6100")
df_2g[["id_cnj", "processo", "classe", "orgao_julgador", "orgao_julgador_colegiado"]]

TRF3 cposg:   0%|          | 0/1 [00:00<?, ?it/s]

TRF3 cposg: 100%|██████████| 1/1 [00:02<00:00,  2.92s/it]

TRF3 cposg: 100%|██████████| 1/1 [00:02<00:00,  2.92s/it]

,id_cnj,processo,classe,orgao_julgador,orgao_julgador_colegiado
0,50211226520184036100,5021122-65.2018.4.03.6100,APELAÇÃO CÍVEL (198),Gab. 13 - DES. FED. MONICA NOBRE,4ª Turma


## Handling processes the public portal cannot return

When a CNJ does not surface in the public consultation (sealed or simply not
found), the row carries only `id_cnj`, with `motivo_falha` set to `None`. A
lookup that fails (network or HTTP error on the search or on `/dados` after
the retries, an unexpected response, a parser error) also yields a row with
only `id_cnj`, but `motivo_falha` names the reason (`timeout`, `conexao`,
`retry_esgotado_<status>`, `resposta_inesperada`), and the failures come in a
single `UserWarning` at the end. An Akamai block stops the batch: the blocked
CNJ and the following ones come back as `bloqueio_anti_bot`, without new
requests, under a `BloqueioAntiBotWarning`.

In [7]:
df_missing = trf3.cpopg("00000000020994030000")
df_missing

TRF3 cpopg:   0%|          | 0/1 [00:00<?, ?it/s]

TRF3 cpopg: 100%|██████████| 1/1 [00:00<00:00, 17.88it/s]

,id_cnj
0,00000000020994030000


## Downloading the documents

With `download_pecas=True`, each listed document is saved as a PDF under
`<diretorio>/<cnj>/<id>.pdf`, and the `pecas` column lists the paths. The
documents that could not be downloaded go to `pecas_falhas`, as
`{"id": ..., "motivo_falha": ...}`.

In [8]:
import tempfile
from pathlib import Path

df_pecas = trf3.cpopg("5003536-21.2025.4.03.6342", download_pecas=True, diretorio=tempfile.mkdtemp())
for caminho in df_pecas.iloc[0]["pecas"]:
    print(Path(caminho).parent.name, Path(caminho).stat().st_size, "bytes")

TRF3 cpopg:   0%|          | 0/1 [00:00<?, ?it/s]

TRF3 cpopg: 100%|██████████| 1/1 [00:04<00:00,  4.84s/it]

TRF3 cpopg: 100%|██████████| 1/1 [00:04<00:00,  4.84s/it]

50035362120254036342 41357 bytes
50035362120254036342 5555 bytes
50035362120254036342 22494 bytes


## Splitting download from parse

`cpopg_download` returns, per CNJ, a dict with the raw JSON envelopes of each
API route (or `None` when the process was not found or the lookup failed,
with the failures in a `UserWarning`); `cpopg_parse` turns that list into the
DataFrame, without the `motivo_falha` column. Useful to cache the raw
responses before processing.

In [9]:
brutos = trf3.cpopg_download("5003536-21.2025.4.03.6342")
print(sorted(brutos[0]))

df_again = trf3.cpopg_parse(brutos, ["50035362120254036342"])
df_again[["id_cnj", "processo", "data_distribuicao"]]

TRF3 cpopg:   0%|          | 0/1 [00:00<?, ?it/s]

TRF3 cpopg: 100%|██████████| 1/1 [00:12<00:00, 12.44s/it]

TRF3 cpopg: 100%|██████████| 1/1 [00:12<00:00, 12.44s/it]

['busca', 'dados', 'documentos', 'movimentacoes', 'outrosInteressados', 'poloAtivo', 'poloPassivo']


,id_cnj,processo,data_distribuicao
0,50035362120254036342,5003536-21.2025.4.03.6342,24/09/2025


## Searching case law (`cjsg`)

`cjsg` searches the TRF3 case-law portal at
[web.trf3.jus.br/jurisprudencia](https://web.trf3.jus.br/jurisprudencia/),
a system separate from the PJe. The `base` argument picks one of the four
result sets the portal keeps apart:

| `base` | Content |
|--------|---------|
| `"acordaos"` (default) | TRF3 appellate judgments |
| `"turmas_recursais"` | judgments of the federal small-claims appeal panels (JEF) |
| `"monocraticas"` | single-judge decisions of the TRF3 |
| `"monocraticas_turmas_recursais"` | single-judge decisions of the appeal panels |

Each call opens its own session on the portal and reads the result list,
which already carries the ementa and the full text of every document: the
cost is three fixed requests (opening the tab, the search POST and the GET of
its redirect, which is page 1) plus one per extra page, with `sleep_time`
between pages.

In [10]:
df_cjsg = trf3.cjsg(
    "medicamento",
    paginas=range(1, 3),
    data_julgamento_inicio="2026-01-01",
    data_julgamento_fim="2026-06-30",
)
print(df_cjsg.shape)
df_cjsg[["processo", "classe", "orgao_julgador", "relator", "data_julgamento", "meio_publicacao"]].head()

Baixando CJSG TRF3:   0%|          | 0/2 [00:00<?, ?it/s]

Baixando CJSG TRF3: 100%|██████████| 2/2 [00:01<00:00,  1.46it/s]

Baixando CJSG TRF3: 100%|██████████| 2/2 [00:01<00:00,  1.46it/s]

(20, 12)


,processo,classe,orgao_julgador,relator,data_julgamento,meio_publicacao
0,5001743-27.2026.4.03.0000,AI - AGRAVO DE INSTRUMENTO,3ª Turma,Desembargador Federal NERY DA COSTA JUNIOR,2026-06-30,DJEN
1,5010057-39.2019.4.03.6100,ApelRemNec - APELAÇÃO / REMESSA NECESSÁRIA,4ª Turma,Desembargador Federal ANDRE NABARRETE NETO,2026-06-30,Intimação via sistema
2,5002047-63.2025.4.03.6110,ApCrim - APELAÇÃO CRIMINAL,5ª Turma,Desembargadora Federal RENATA ANDRADE LOTUFO,2026-06-30,Intimação via sistema
3,5002192-62.2020.4.03.6121,ApCiv - APELAÇÃO CÍVEL,4ª Turma,Desembargador Federal ANDRE NABARRETE NETO,2026-06-30,DJEN
4,5004682-95.2021.4.03.6000,ApCiv - APELAÇÃO CÍVEL,Turma Regional de Mato Grosso do Sul,Juiz Federal Convocado NEY GUSTAVO PAES DE AND...,2026-06-30,Intimação via sistema


### The other bases

In the single-judge bases, `ementa` holds the text of the decision (there is
no ementa) and `orgao_julgador` is empty; `data_julgamento` is the date of the
decision.

In part of the older appeal-panel judgments (`turmas_recursais`), `ementa` is
null because the portal sends an empty ementa block; `inteiro_teor` still
carries the full text. When this was measured, every ementa from 2013 and 2014
was empty, and some from 2011.

In [11]:
for base in ["turmas_recursais", "monocraticas", "monocraticas_turmas_recursais"]:
    df_base = trf3.cjsg(
        "medicamento",
        base=base,
        paginas=1,
        data_julgamento_inicio="2026-01-01",
        data_julgamento_fim="2026-06-30",
    )
    print(base, df_base.shape, df_base["relator"].iloc[0])

turmas_recursais (10, 12) Juíza Federal MAIRA FELIPE LOURENCO


monocraticas (10, 12) Desembargadora Federal GISELLE DE AMARO E FRANCA


monocraticas_turmas_recursais (10, 12) Juíza Federal LIN PEI JENG


### Filters

`relator` and `classe` are free text matched against the
portal's index, not dropdown IDs. Use the spelling of the form's dropdowns:
`relator="NERY JUNIOR"`, `classe="AI - AGRAVO DE INSTRUMENTO"`.

`orgao_julgador` picks one option of the órgão dropdown of the tab, by its
label (`"3ª Turma"`, `"11ª Turma Recursal da Seção Judiciária de São Paulo"`)
or by its indexed name (`"TERCEIRA TURMA"`, `"11ª TURMA RECURSAL DE SÃO
PAULO"`), ignoring case and spacing. The value in the `orgao_julgador` column
also works: as in the portal's form, `"1a. seção"` becomes `"1A"`, so `"QUINTA
TURMA - 1A. SEÇÃO"` matches the option `QUINTA TURMA - 1a. SEÇÃO`. The search
reads the dropdown from the tab it already opens and sends the pair the browser
sends, the label and the indexed name of the option, and the portal returns the
documents recorded under either (measured with "medicamento": `"QUINTA TURMA"`
finds 576 documents, against 443 for the label alone and 133 for the indexed
name alone). A label shared by more than one option, such as `"5ª Turma"` in
the acórdãos base, raises `ValueError` listing the indexed names that resolve
it; text that matches no option is sent as is and only finds what the portal
records with that spelling. The monocratic bases have no órgão filter.

The form has a single date pair, so pass either `data_julgamento_*` or
`data_publicacao_*`, not both.

In [12]:
df_filtrado = trf3.cjsg(
    "medicamento",
    relator="NERY JUNIOR",
    classe="AI - AGRAVO DE INSTRUMENTO",
    data_publicacao_inicio="2026-01-01",
    data_publicacao_fim="2026-06-30",
    paginas=1,
)
df_filtrado[["processo", "relator", "relator_acordao", "data_publicacao"]]

,processo,relator,relator_acordao,data_publicacao
0,5027413-04.2025.4.03.0000,Desembargador Federal NERY DA COSTA JUNIOR,None,2026-06-25
